In [ ]:
!pip install -U bitsandbytes peft
!install transformers==4.38.0

In [ ]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

# Retrieve the secret
user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_Token")

# Log in
login(token=hf_token)

In [ ]:
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

After running the cell above and authenticating, your Google Drive will be mounted at `/content/drive`. Now, let's navigate to your specified folder and list its contents to confirm access.

In [ ]:
folder_path = "/kaggle/input/datasets/bhavyranka/dataset"

In [ ]:
import pandas as pd
import json

# Define the full paths to the JSON files
fft_file_path = os.path.join(folder_path, 'cwru_fft_dataset.json')
stat_file_path = os.path.join(folder_path, 'cwru_stat_dataset.json')

# Load cwru_fft_dataset.json
try:
    with open(fft_file_path, 'r') as f:
        fft_data = json.load(f)
    fft_df = pd.DataFrame(fft_data)
    print(f"Successfully loaded {fft_file_path}")
    print("First 5 rows of cwru_fft_dataset.json:")
    display(fft_df.head())
except FileNotFoundError:
    print(f"Error: {fft_file_path} not found.")
except Exception as e:
    print(f"An error occurred loading {fft_file_path}: {e}")

In [ ]:
# Load cwru_stat_dataset.json
try:
    with open(stat_file_path, 'r') as f:
        stat_data = json.load(f)
    stat_df = pd.DataFrame(stat_data)
    print(f"Successfully loaded {stat_file_path}")
    print("First 5 rows of cwru_stat_dataset.json:")
    display(stat_df.head())
except FileNotFoundError:
    print(f"Error: {stat_file_path} not found.")
except Exception as e:
    print(f"An error occurred loading {stat_file_path}: {e}")

In [ ]:
import os
import json
import argparse
import random
import logging
from typing import Optional

# Added for Colab/Data Handling
import pandas as pd
from google.colab import drive

# --- DRIVE MOUNTING ---
# Uncomment the line below if you haven't mounted your drive yet
# drive.mount('/content/drive')

import numpy as np
import torch
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, classification_report
)
from sklearn.model_selection import train_test_split

import transformers
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    DataCollatorForSeq2Seq,
    BitsAndBytesConfig,
    IntervalStrategy
)
from peft import (
    LoraConfig,
    get_peft_model,
    TaskType,
    PeftModel,
)
from datasets import Dataset

FOLDER_PATH = '/kaggle/input/datasets/bhavyranka/dataset'

def verify_and_load_data():
    """Checks for folder existence and prints data previews."""
    if os.path.exists(FOLDER_PATH):
        print(f"Successfully accessed: {FOLDER_PATH}")
        print("Contents of the folder:", os.listdir(FOLDER_PATH))
    else:
        print(f"Error: Folder not found at {FOLDER_PATH}")
        return

    # Preview logic for verification
    fft_path = os.path.join(FOLDER_PATH, 'cwru_fft_dataset.json')
    try:
        with open(fft_path, 'r') as f:
            data = json.load(f)
        print(f"\nSuccessfully loaded preview of {fft_path}")
        print(pd.DataFrame(data).head())
    except Exception as e:
        print(f"Preview failed: {e}")

# ─────────────────────────────────────────────
# Logging & Constants
# ─────────────────────────────────────────────
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    datefmt="%H:%M:%S",
)
logger = logging.getLogger(__name__)


In [ ]:
FAULT_LABELS = ["NO", "IRF", "ORF", "REF"]   # Normal, Inner Race, Outer Race, Rolling Element
LABEL2ID = {l: i for i, l in enumerate(FAULT_LABELS)}
ID2LABEL = {i: l for l, i in LABEL2ID.items()}

DEFAULT_MODEL = "meta-llama/Meta-Llama-3-8B-Instruct"   # swap for any HF model id
LORA_RANK     = 4          # paper: LoRA rank 4
LORA_ALPHA    = 16
LORA_DROPOUT  = 0.05
LEARNING_RATE = 1e-4       # paper: 1e-4
BATCH_SIZE    = 2          # paper: batch size 2
NUM_EPOCHS    = 3          # paper: 3 epochs
MAX_SEQ_LEN   = 1024       # safe for stat data; FFT may need more


def build_prompt(instruction: str, input_text: str, output: str = "") -> str:
    """
    Alpaca-style instruction prompt used by the paper.
    If output is empty the closing tag is omitted (for inference).
    """
    prompt = (
        f"### Instruction:\n{instruction}\n\n"
        f"### Input:\n{input_text}\n\n"
        f"### Response:\n"
    )
    if output:
        prompt += output
    return prompt



In [ ]:
def build_inference_prompt(instruction: str, input_text: str) -> str:
    return build_prompt(instruction, input_text, output="")

In [ ]:

def load_dataset_json(path: str):
    with open(path, "r") as f:
        data = json.load(f)
    logger.info(f"Loaded {len(data)} samples from {path}")

    
    unknown = {d["output"] for d in data if d["output"] not in LABEL2ID}
    if unknown:
        logger.warning(f"Unknown labels found: {unknown}. They will be ignored during eval.")

    return data


def split_data(data, test_size=0.10, seed=42):
    """90/10 train-test split (paper: 10% for evaluation)."""
    train, test = train_test_split(data, test_size=test_size, random_state=seed,
                                   stratify=[d["output"] for d in data])
    logger.info(f"Train: {len(train)} | Test: {len(test)}")
    return train, test


def label_distribution(data, split_name=""):
    from collections import Counter
    dist = Counter(d["output"] for d in data)
    logger.info(f"Label distribution [{split_name}]: {dict(dist)}")


def tokenize_sample(sample, tokenizer, max_seq_len: int):
    full_text  = build_prompt(sample["instruction"], sample["input"], sample["output"])
    prompt_only = build_inference_prompt(sample["instruction"], sample["input"])

    full_ids   = tokenizer(full_text,   truncation=True, max_length=max_seq_len)["input_ids"]
    prompt_ids = tokenizer(prompt_only, truncation=True, max_length=max_seq_len)["input_ids"]

    labels = [-100] * len(prompt_ids) + full_ids[len(prompt_ids):]

    return {
        "input_ids":      full_ids,
        "attention_mask": [1] * len(full_ids),
        "labels":         labels,
    }


In [ ]:
def make_hf_dataset(data, tokenizer, max_seq_len: int) -> Dataset:
    tokenized = [tokenize_sample(s, tokenizer, max_seq_len) for s in data]
    return Dataset.from_list(tokenized)

In [ ]:
def load_model_and_tokenizer(model_id: str, load_in_4bit: bool = True):
    tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
        tokenizer.pad_token_id = tokenizer.eos_token_id

    bnb_config = None
    if load_in_4bit:
        try:
            import bitsandbytes  
            bnb_config = BitsAndBytesConfig(
                load_in_4bit=True,
                bnb_4bit_quant_type="nf4",
                bnb_4bit_compute_dtype=torch.bfloat16,
                bnb_4bit_use_double_quant=True,
            )
            logger.info("4-bit quantisation enabled (bitsandbytes).")
        except ImportError:
            logger.warning("bitsandbytes not found – loading in float16 instead.")

    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        quantization_config=bnb_config,
        torch_dtype=torch.bfloat16 if bnb_config is None else None,
        device_map="auto",
        trust_remote_code=True,
    )
    model.config.use_cache = False   
    return model, tokenizer




In [ ]:
def apply_lora(model, lora_rank: int = LORA_RANK):
    """Wrap model with LoRA adapters (paper: rank 4, target q/v projections)."""
    lora_config = LoraConfig(
        task_type=TaskType.CAUSAL_LM,
        r=lora_rank,
        lora_alpha=LORA_ALPHA,
        lora_dropout=LORA_DROPOUT,
        bias="none",
        target_modules=["q_proj", "v_proj"],   
    )
    model = get_peft_model(model, lora_config)
    model.print_trainable_parameters()
    return model

In [ ]:
def train(model, tokenizer, train_data, eval_data, output_dir: str,
          max_seq_len: int, num_epochs: int, batch_size: int, lr: float):

    train_ds = make_hf_dataset(train_data, tokenizer, max_seq_len)
    eval_ds  = make_hf_dataset(eval_data,  tokenizer, max_seq_len)

    
    training_args = TrainingArguments(
        output_dir=output_dir,
        num_train_epochs=num_epochs,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=batch_size,
        learning_rate=lr,
        lr_scheduler_type="cosine",
        bf16=torch.cuda.is_bf16_supported(),
        fp16=not torch.cuda.is_bf16_supported() and torch.cuda.is_available(),
        gradient_accumulation_steps=4,
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        logging_steps=50,
        report_to="none",
        dataloader_num_workers=0,
    )

    data_collator = DataCollatorForSeq2Seq(
        tokenizer, model=model, padding=True, pad_to_multiple_of=8
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_ds,
        eval_dataset=eval_ds,
        data_collator=data_collator,
    )

    logger.info("Starting fine-tuning …")
    trainer.train()

    
    final_dir = os.path.join(output_dir, "checkpoint-final")
    model.save_pretrained(final_dir)
    tokenizer.save_pretrained(final_dir)
    logger.info(f"Model saved to {final_dir}")

    return trainer

In [ ]:
def extract_label(generated_text: str) -> str:

    text = generated_text.strip().upper()
    for label in FAULT_LABELS:
        if label in text:
            return label
    # Fallback: return first word
    first_word = text.split()[0] if text else "UNKNOWN"
    return first_word


@torch.no_grad()
def run_inference(model, tokenizer, samples, max_new_tokens: int = 16,
                  batch_size: int = 8) -> list[str]:
    model.eval()
    predictions = []

    for i in range(0, len(samples), batch_size):
        batch = samples[i: i + batch_size]
        prompts = [
            build_inference_prompt(s["instruction"], s["input"]) for s in batch
        ]

        inputs = tokenizer(
            prompts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=MAX_SEQ_LEN,
        ).to(model.device)

        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,         
            temperature=1.0,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

      
        generated = outputs[:, inputs["input_ids"].shape[1]:]
        decoded   = tokenizer.batch_decode(generated, skip_special_tokens=True)
        predictions.extend([extract_label(d) for d in decoded])

        if (i // batch_size) % 10 == 0:
            logger.info(f"  Inference: {min(i + batch_size, len(samples))}/{len(samples)}")

    return predictions

In [ ]:
def evaluate(predictions: list[str], ground_truth: list[str], label_names=None):
    """
    Compute Accuracy, Precision, Recall, F1-Score, and Confusion Matrix.
    """
    if label_names is None:
        label_names = sorted(set(ground_truth))

    valid_pairs = [(p, t) for p, t in zip(predictions, ground_truth)
                   if p in FAULT_LABELS and t in FAULT_LABELS]
    if len(valid_pairs) < len(predictions):
        n_invalid = len(predictions) - len(valid_pairs)
        logger.warning(f"{n_invalid} predictions were unmappable and treated as wrong.")

        
        predictions = [p if p in FAULT_LABELS else "__INVALID__"
                       for p in predictions]
        valid_pairs = list(zip(predictions, ground_truth))

    preds, truths = zip(*valid_pairs) if valid_pairs else ([], [])

    acc  = accuracy_score(truths, preds)
    prec = precision_score(truths, preds, labels=label_names,
                           average="weighted", zero_division=0)
    rec  = recall_score(truths, preds, labels=label_names,
                        average="weighted", zero_division=0)
    f1   = f1_score(truths, preds, labels=label_names,
                    average="weighted", zero_division=0)
    cm   = confusion_matrix(truths, preds, labels=label_names)

    print("\n" + "="*60)
    print("  EVALUATION RESULTS")
    print("="*60)
    print(f"  Accuracy  : {acc:.4f}")
    print(f"  Precision : {prec:.4f}")
    print(f"  Recall    : {rec:.4f}")
    print(f"  F1-Score  : {f1:.4f}")
    print("\n  Classification Report:")
    print(classification_report(truths, preds, labels=label_names,
                                zero_division=0))
    print("  Confusion Matrix:")
    header = "       " + "  ".join(f"{l:>5}" for l in label_names)
    print(header)
    for i, row in enumerate(cm):
        row_str = "  ".join(f"{v:5d}" for v in row)
        print(f"  {label_names[i]:>5}  {row_str}")
    print("="*60 + "\n")

    return {
        "accuracy":  acc,
        "precision": prec,
        "recall":    rec,
        "f1":        f1,
        "confusion_matrix": cm.tolist(),
    }



In [ ]:
def parse_args():
    parser = argparse.ArgumentParser(description="FD-LLM fine-tuning and evaluation")

    # Data
    parser.add_argument("--data_path",  type=str, required=True,
                        help="Path to training JSON (cwru_stat_dataset.json or cwru_fft_dataset.json)")
    parser.add_argument("--data_type",  type=str, choices=["stat", "fft"], required=True,
                        help="Dataset type: 'stat' for statistical features, 'fft' for FFT-processed")
    parser.add_argument("--cross_eval_path", type=str, default=None,
                        help="Optional: path to a separate JSON for cross-dataset evaluation")
    parser.add_argument("--test_size",  type=float, default=0.10,
                        help="Fraction of data held out for evaluation (default: 0.10)")

    # Model
    parser.add_argument("--model_id",   type=str, default=DEFAULT_MODEL,
                        help="HuggingFace model ID (default: meta-llama/Meta-Llama-3-8B-Instruct)")
    parser.add_argument("--load_in_4bit", action="store_true", default=True,
                        help="Load model in 4-bit precision (bitsandbytes)")
    parser.add_argument("--lora_rank",  type=int, default=LORA_RANK)
    parser.add_argument("--max_seq_len", type=int, default=MAX_SEQ_LEN)

    # Training
    parser.add_argument("--output_dir", type=str, default="./fd_llm_output")
    parser.add_argument("--num_epochs", type=int, default=NUM_EPOCHS)
    parser.add_argument("--batch_size", type=int, default=BATCH_SIZE)
    parser.add_argument("--lr",         type=float, default=LEARNING_RATE)
    parser.add_argument("--seed",       type=int, default=42)

    # Modes
    parser.add_argument("--eval_only",  action="store_true",
                        help="Skip training; load checkpoint and evaluate only")
    parser.add_argument("--checkpoint_dir", type=str, default=None,
                        help="Path to a saved LoRA checkpoint for --eval_only")

    return parser.parse_args()


In [ ]:
import sys
def main():
    original_argv = sys.argv
    try:
        sys.argv = [
            'colab_script.py', # A dummy script name
            '--data_path', os.path.join(FOLDER_PATH, 'cwru_stat_dataset.json'),
            '--data_type', 'stat',
            '--output_dir', os.path.join('/kaggle/working/fd_llm_output_colab', 'fd_llm_output_colab') # Example output directory within your drive
        ]
        args = parse_args()
    finally:
        sys.argv = original_argv

    random.seed(args.seed)
    np.random.seed(args.seed)
    torch.manual_seed(args.seed)

    logger.info(f"Data type   : {args.data_type.upper()}")
    logger.info(f"Model       : {args.model_id}")
    logger.info(f"LoRA rank   : {args.lora_rank}")
    logger.info(f"Max seq len : {args.max_seq_len}")

    data = load_dataset_json(args.data_path)
    label_distribution(data, "full")

    train_data, test_data = split_data(data, test_size=args.test_size, seed=args.seed)
    label_distribution(train_data, "train")
    label_distribution(test_data,  "test")

    # ── Load model ─────────────────────────────
    model, tokenizer = load_model_and_tokenizer(args.model_id, args.load_in_4bit)

    if args.eval_only:
        # ── Eval-only mode ─────────────────────
        if args.checkpoint_dir is None:
            args.checkpoint_dir = os.path.join(args.output_dir, "checkpoint-final")
        logger.info(f"Loading LoRA checkpoint from {args.checkpoint_dir}")
        model = PeftModel.from_pretrained(model, args.checkpoint_dir)

    else:
        model = apply_lora(model, lora_rank=args.lora_rank)

        train(
            model=model,
            tokenizer=tokenizer,
            train_data=train_data,
            eval_data=test_data,
            output_dir=args.output_dir,
            max_seq_len=args.max_seq_len,
            num_epochs=args.num_epochs,
            batch_size=args.batch_size,
            lr=args.lr,
        )

    logger.info("\n--- In-distribution evaluation ---")
    preds  = run_inference(model, tokenizer, test_data)
    truths = [d["output"] for d in test_data]
    results = evaluate(preds, truths, label_names=FAULT_LABELS)

    os.makedirs(args.output_dir, exist_ok=True)
    results_path = os.path.join(
        args.output_dir,
        f"results_{args.data_type}_indist.json"
    )
    with open(results_path, "w") as f:
        json.dump(results, f, indent=2)
    logger.info(f"Results saved to {results_path}")


    if args.cross_eval_path:
        logger.info("\n--- Cross-dataset evaluation ---")
        cross_data = load_dataset_json(args.cross_eval_path)
        label_distribution(cross_data, "cross-eval")
        cross_preds  = run_inference(model, tokenizer, cross_data)
        cross_truths = [d["output"] for d in cross_data]
        cross_results = evaluate(cross_preds, cross_truths, label_names=FAULT_LABELS)

        cross_path = os.path.join(
            args.output_dir,
            f"results_{args.data_type}_cross.json"
        )
        with open(cross_path, "w") as f:
            json.dump(cross_results, f, indent=2)
        logger.info(f"Cross-eval results saved to {cross_path}")


if __name__ == "__main__":
    main()

In [ ]:
import shutil
shutil.make_archive('my_model_output', 'zip', '/kaggle/working/fd_llm_output_colab')